In [ ]:
from collections import defaultdict

groups = {
    'Phe': ['TTT', 'TTC'],
    'Leu': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'Ile': ['ATT', 'ATC', 'ATA'],
    'Met': ['ATG'],
    'Val': ['GTT', 'GTC', 'GTA', 'GTG'],
    'Ser': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
    'Pro': ['CCT', 'CCC', 'CCA', 'CCG'],
    'Thr': ['ACT', 'ACC', 'ACA', 'ACG'],
    'Ala': ['GCT', 'GCC', 'GCA', 'GCG'],
    'Tyr': ['TAT', 'TAC'],
    'End': ['TAA', 'TAG', 'TGA'],
    'His': ['CAT', 'CAC'],
    'Gln': ['CAA', 'CAG'],
    'Asn': ['AAT', 'AAC'],
    'Lys': ['AAA', 'AAG'],
    'Asp': ['GAT', 'GAC'],
    'Glu': ['GAA', 'GAG'],
    'Cys': ['TGT', 'TGC'],
    'Trp': ['TGG'],
    'Arg': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'Gly': ['GGT', 'GGC', 'GGA', 'GGG']
}

codon_table = {codon: aa for aa, codons in groups.items() for codon in codons}

def clean_sequence(seq):
    return ''.join(filter(lambda c: c in 'ACGTacgt', seq.upper()))

def get_codon_usage(sequence, seq_name="sample sequence one"):
    usage = defaultdict(int)
    cleaned = clean_sequence(sequence)

    for i in range(0, len(cleaned) - 2, 3):
        codon = cleaned[i:i+3]
        if len(codon) == 3:
            usage[codon] += 1

    total_codons = sum(usage.values())
    if total_codons == 0:
        print(f"\nNo valid codons found for {seq_name}.")
        return

    aa_to_codons = defaultdict(list)
    for codon in sorted(codon_table.keys()):
        aa = codon_table[codon]
        aa_to_codons[aa].append(codon)

    print(f"\nResults for {total_codons} residue sequence \"{seq_name}\" starting \"{cleaned[:9].lower()}\"\n")
    print("AmAcid   Codon     Number        /1000     Fraction   ..\n")

    for aa in sorted(aa_to_codons.keys()):
        for codon in aa_to_codons[aa]:
            count = usage.get(codon, 0)
            per_thousand = (count / total_codons) * 1000
            related_codons = aa_to_codons[aa]
            total_for_aa = sum(usage.get(c, 0) for c in related_codons)
            fraction = (count / total_for_aa) if total_for_aa else 0.0

            print(f"{aa:<8} {codon:<8} {count:6.2f} {per_thousand:10.2f} {fraction:12.2f}")

if __name__ == "__main__":
    print("=== Codon Usage Analyzer ===")
    print("Paste the raw sequence of one or more FASTA sequences into the text area. Input limit is 500,000,000 characters (and press ENTER)")

    data = []
    while True:
        try:
            line = input()
            if line.strip() == "":
                break
            data.append(line.strip())
        except EOFError:
            break

    # Merge all lines into one big string
    joined_data = " ".join(data)

    # Split at each '>' to separate sequences
    records = joined_data.split('>')
    records = [r.strip() for r in records if r.strip()]

    if not records:
        print("No sequences entered.")
    else:
        for record in records:
            if ' ' in record:
                header, sequence = record.split(' ', 1)
            else:
                header = record
                sequence = ''
            sequence = sequence.replace(' ', '').replace('\n', '')
            if sequence:
                get_codon_usage(sequence, header)
            else:
                print(f"\nNo sequence found for {header}.")


=== Codon Usage Analyzer ===
Paste the raw sequence of one or more FASTA sequences into the text area. Input limit is 500,000,000 characters (and press ENTER)
>sample sequence one atgtttcagaaagaggatcttgctacatggatgcaaatttcaatgagtggtcaatttgatgatacagcattagaggaatggagtacaaatggtaaagaacctgagatctgtgagaaatctccaaaagctgatggagttactacgattatggagagagctctatgtccatgggatagcagagtcaactaccaagagagccgagaacctaaattgattgctgaatcagtttgtctatgccgtaagagccgtggttctacaggagctttctgtatgccaattgttcgaaaagttccaattctccgacgtgtctcttgtgatcgttcaacaggtttatggaattatgtaagatcaactgagctaataactgttggatgtcattctgtattgccaagaactcaaagagcagcacgtcttgcccatttatcatcttctcgtattattgtttaa  >sample sequence two caggttacaattggagccatttcatcttctgactgaggaaatcaggtccgacagcgtagatgtatacacgctctattcacaaaattggtaacgattctg


Results for 149 residue sequence "sample" starting "catgtttca"

AmAcid   Codon     Number        /1000     Fraction   ..

Ala      GCA        1.00       6.71         0.25
Ala      GCC        2.00      13.42         0.50
Ala      GCG        0.00    